# Locate One Sample Across Azure and Neptune

This notebook answers the handover question: for one `WAVEANYYYYMMDD` sample, which artifact versions exist on Azure and which exist on Neptune?

Use it first. It gives the exact path for raw `.nc`, engineered `.parquet`, and preprocessed `.pt` where available.

In [1]:
from pathlib import Path
import sys

candidate_dirs = [
    Path.cwd(),
    Path.cwd() / 'notebooks' / 'handover',
]
for candidate in candidate_dirs:
    if (candidate / 'sample_data_helpers.py').exists():
        sys.path.insert(0, str(candidate))
        break

from sample_data_helpers import locate_sample_files

In [2]:
# Parameters
SAMPLE_DATE = '20230101'

# Optional per-environment overrides if your mounts/servers differ.
OVERRIDES_BY_ENV = {
    'azure': {},
    'neptune': {},
}

In [3]:
df = locate_sample_files(SAMPLE_DATE, overrides_by_env=OVERRIDES_BY_ENV)
df

,environment,artifact_key,description,path,exists
0,azure,nc_raw_uncorrected,Raw degraded model data (.nc),None,False
1,azure,nc_reference,Raw reference / corrected data (.nc),None,False
2,azure,parquet_engineered,Engineered modeling table (.parquet),/mnt/blobstorage/parquet/hourly_extra_features...,False
3,azure,pt_preprocessed,Training-ready preprocessed tensor (.pt),/mnt/local_datasets/preprocessed_extended_subs...,False
4,azure,pt_preprocessed_blob,Blob-backed preprocessed tensor (.pt),/mnt/blobstorage/preprocessed_extended_subsamp...,False
5,neptune,nc_raw_uncorrected,Raw degraded model data (.nc),/data/tsolis/AI_project/without_reduced/WAVEAN...,False
6,neptune,nc_reference,Raw reference / corrected data (.nc),/data/tsolis/AI_project/with_reduced/WAVEAN202...,False
7,neptune,parquet_engineered,Engineered modeling table (.parquet),/data/tsolis/AI_project/parquet/augmented_with...,False
8,neptune,pt_preprocessed,Training-ready preprocessed tensor (.pt),/data/tsolis/AI_project/preprocessed_subsample...,False
9,neptune,pt_preprocessed_blob,Blob-backed preprocessed tensor (.pt),None,False


## How To Read This

- `nc_raw_uncorrected`: degraded model raw `.nc`
- `nc_reference`: reference / corrected raw `.nc`
- `parquet_engineered`: engineered modeling table
- `pt_preprocessed`: training-ready tensor file

Current documented workflow: start from Azure-mounted paths when available. Use Neptune when you need older/internal raw data or when Azure does not contain the representation you need.